# 03 IFC + BCF Viewer (V1 Prototype)

Interaktiver V1-Workflow: IFC laden, BCF laden, Topic auswählen, GUIDs hervorheben.


In [ ]:
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

from openbim_viewer.viewer_adapter import IFCViewerAdapter
from openbim_viewer.ifc_loader import load_ifc, build_ifc_index
from openbim_viewer.bcf_loader import extract_bcf_topics, extract_bcf_viewpoints
from openbim_viewer.mapping import map_bcf_to_ifc

def detect_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'src').exists() and (cwd / 'data').exists():
        return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'src').exists():
        return cwd.parent
    for p in [cwd, *cwd.parents]:
        if (p / 'src').exists() and (p / 'data').exists():
            return p
    return cwd

PROJECT_ROOT = detect_project_root()
DEFAULT_IFC_PATH = PROJECT_ROOT / 'data' / 'sample.ifc'
DEFAULT_BCF_PATH = PROJECT_ROOT / 'data' / 'sample.bcfzip'

print(f'PROJECT_ROOT: {PROJECT_ROOT.resolve()}')
print(f'DEFAULT_IFC_PATH: {DEFAULT_IFC_PATH.resolve()}')
print(f'DEFAULT_BCF_PATH: {DEFAULT_BCF_PATH.resolve()}')


In [ ]:
ifc_path_input = widgets.Text(description='IFC_PATH', value=str(DEFAULT_IFC_PATH), layout=widgets.Layout(width='900px'))
bcf_path_input = widgets.Text(description='BCF_PATH', value=str(DEFAULT_BCF_PATH), layout=widgets.Layout(width='900px'))

btn_load_ifc = widgets.Button(description='Load IFC', button_style='primary')
btn_load_bcf = widgets.Button(description='Load BCF', button_style='info')
btn_build_mapping = widgets.Button(description='Build Mapping')

topic_dropdown = widgets.Dropdown(description='BCF Topic', options=[], layout=widgets.Layout(width='900px'))

out_topic = widgets.Output(layout=widgets.Layout(border='1px solid #ddd'))
out_viewer = widgets.Output(layout=widgets.Layout(border='1px solid #ddd'))
out_diag = widgets.Output(layout=widgets.Layout(border='1px solid #eee'))

ui = widgets.VBox([
    ifc_path_input,
    bcf_path_input,
    widgets.HBox([btn_load_ifc, btn_load_bcf, btn_build_mapping]),
    topic_dropdown,
    widgets.HTML('<b>Topic Info</b>'),
    out_topic,
    widgets.HTML('<b>Viewer</b>'),
    out_viewer,
    widgets.HTML('<b>Mapping / Diagnose</b>'),
    out_diag,
])
display(ui)


In [ ]:
adapter = IFCViewerAdapter(max_elements=200)
viewer_widget = None
ifc_file = None
ifc_index = {}
topics = []
viewpoints = []
df_mapping = None
topic_guid_to_topic = {}
topic_guid_to_guids = {}

def log(msg: str):
    with out_diag:
        print(msg)

def build_mapping_now():
    global df_mapping, topic_guid_to_guids
    if not topics:
        log('Hinweis: Keine Topics geladen. Bitte zuerst BCF laden.')
        return
    if not ifc_index:
        log('Hinweis: IFC ist noch nicht geladen. Mapping nutzt dann leeren IFC-Index.')

    df_mapping = map_bcf_to_ifc(topics, viewpoints, ifc_index)
    topic_guid_to_guids = {}

    if df_mapping is not None and not df_mapping.empty:
        for tg, g in df_mapping.groupby('topic_guid', dropna=False):
            if tg is None:
                continue
            guids = [x for x in g['referenced_ifc_guid'].dropna().tolist() if x]
            topic_guid_to_guids[tg] = list(dict.fromkeys(guids))

    log(f'Mapping gebaut: {0 if df_mapping is None else len(df_mapping)} Zeilen')
    if df_mapping is not None and not df_mapping.empty:
        with out_diag:
            display(df_mapping.head(20))

def on_load_ifc(_):
    global ifc_file, ifc_index, viewer_widget
    out_diag.clear_output()
    out_viewer.clear_output()

    p = Path(ifc_path_input.value).expanduser()
    if not p.exists():
        log(f'Bitte eine IFC-Datei unter data/sample.ifc ablegen oder IFC_PATH anpassen. Aktuell nicht gefunden: {p}')
        return

    try:
        ifc_file = load_ifc(p)
        ifc_index = build_ifc_index(ifc_file)
        viewer_widget = adapter.show(p)
        caps = adapter.capabilities()
        with out_viewer:
            display(viewer_widget)

        log(f'IFC geladen: {p}')
        log(f'IfcProduct im Index: {len(ifc_index)}')
        log(f'Viewer backend={caps.backend}, can_render={caps.can_render}, can_highlight_guid={caps.can_highlight_guid}')
    except Exception as exc:
        log(f'Fehler beim IFC-Laden: {exc}')

def on_load_bcf(_):
    global topics, viewpoints, topic_guid_to_topic
    out_diag.clear_output()
    out_topic.clear_output()

    p = Path(bcf_path_input.value).expanduser()
    if not p.exists():
        log(f'Bitte eine BCF-Datei unter data/sample.bcfzip ablegen oder BCF_PATH anpassen. Aktuell nicht gefunden: {p}')
        return

    try:
        topics = extract_bcf_topics(p)
        viewpoints = extract_bcf_viewpoints(p)
        topic_guid_to_topic = {t.get('topic_guid'): t for t in topics if t.get('topic_guid')}

        log(f'BCF geladen: {p}')
        log(f'Topics: {len(topics)}, Viewpoints: {len(viewpoints)}')

        build_mapping_now()

        options = []
        for t in topics:
            tg = t.get('topic_guid')
            if not tg:
                continue
            title = t.get('title') or '(ohne Titel)'
            status = t.get('status') or '(ohne Status)'
            guid_count = len(topic_guid_to_guids.get(tg, []))
            options.append((f'{title} | {status} | {guid_count} GUIDs', tg))

        topic_dropdown.options = options
        if not options:
            log('Hinweis: Keine Topic-GUIDs in der BCF gefunden.')
    except Exception as exc:
        log(f'Fehler beim BCF-Laden: {exc}')

def on_build_mapping(_):
    out_diag.clear_output()
    build_mapping_now()

def on_topic_change(change):
    if change.get('name') != 'value' or change.get('new') is None:
        return

    topic_guid = change['new']
    topic = topic_guid_to_topic.get(topic_guid, {})
    guids = topic_guid_to_guids.get(topic_guid, [])

    found = [g for g in guids if g in ifc_index]
    missing = [g for g in guids if g not in ifc_index]

    with out_topic:
        out_topic.clear_output()
        print(f"Title: {topic.get('title') or '(ohne Titel)'}")
        print(f"Status: {topic.get('status') or '(ohne Status)'}")
        print(f"Type: {topic.get('type') or '(ohne Typ)'}")
        print(f"Priority: {topic.get('priority') or '(ohne Priorität)'}")
        print(f"Author: {topic.get('creation_author') or '(ohne Autor)'}")
        print('Kommentare:')
        comments = topic.get('comments') or []
        if comments:
            for c in comments:
                print(f' - {c}')
        else:
            print(' - (keine)')

        print('GUIDs:')
        if guids:
            for g in guids:
                print(f' - {g}')
        else:
            print(' - (keine)')

        print(f'Gefunden im IFC: {len(found)}')
        for g in found:
            print(f' + {g}')
        print(f'Nicht im IFC gefunden: {len(missing)}')
        for g in missing:
            print(f' - {g}')

    if not guids:
        log('Hinweis: Dieses Topic referenziert keine GUIDs.')

    if adapter.viewer is None:
        log('Hinweis: Viewer nicht initialisiert. Bitte zuerst IFC laden.')
        return

    try:
        adapter.clear_highlight()
    except Exception as exc:
        log(f'Warnung: clear_highlight fehlgeschlagen: {exc}')

    caps = adapter.capabilities()
    if not caps.can_highlight_guid:
        log('Viewer unterstützt highlight_guids nicht.')
        return

    try:
        adapter.highlight_guids(guids)
        if guids:
            log(f'Highlighting angefordert für {len(guids)} GUID(s).')
    except Exception as exc:
        log(f'Warnung: highlight_guids fehlgeschlagen: {exc}')

    if guids and not found:
        log('Hinweis: Keine der Topic-GUIDs wurde im IFC-Index gefunden.')

btn_load_ifc.on_click(on_load_ifc)
btn_load_bcf.on_click(on_load_bcf)
btn_build_mapping.on_click(on_build_mapping)
topic_dropdown.observe(on_topic_change, names='value')
